# 中芯国际 AH 股行情数据获取与可视化分析

**A股代码**: 688981.SH（科创板）  
**港股代码**: 00981.HK（中国香港）  

本 Notebook 演示完整的量化数据分析流程：
1. 通过 **Tushare Pro API** 获取 A 股日线行情
2. 通过 **腾讯行情 API** 获取港股日线行情（前复权）
3. 数据清洗、对齐与 AH 溢价率计算
4. 使用 **mplfinance + matplotlib** 绘制 K 线图、成交量图
5. 归一化价格走势对比与 AH 溢价率分析

> 配色遵循中国股市惯例：**涨红跌绿**

## 一、环境准备

In [ ]:
import tushare as ts
import pandas as pd
import requests
import numpy as np
import mplfinance as mpf
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime, timedelta

# 中文字体设置
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

# 中国股市配色：涨红跌绿
UP_COLOR = '#ef4444'
DOWN_COLOR = '#22c55e'

print('环境准备完成')

## 二、配置参数

请替换为你的 Tushare Token（获取地址：https://tushare.pro → 个人中心 → MCP Server）

In [ ]:
TUSHARE_TOKEN = '你的TushareToken'  # 替换为你的 Token

# 港币兑人民币汇率（近似值）
HKD_CNY_RATE = 0.92

# 数据区间：近一年
END_DATE = datetime.now().strftime('%Y%m%d')
START_DATE = (datetime.now() - timedelta(days=365)).strftime('%Y%m%d')
print(f'数据区间: {START_DATE} ~ {END_DATE}')

## 三、获取中芯国际 A 股日线数据

数据来源：**Tushare Pro API** `pro.daily()` 接口  
字段说明：开高低收、前收价、涨跌额、涨跌幅、成交量（手）、成交额（千元）

In [ ]:
# 获取 A 股日线数据
pro = ts.pro_api(TUSHARE_TOKEN)

df_a = pro.daily(
    ts_code='688981.SH',
    start_date=START_DATE,
    end_date=END_DATE,
    fields='ts_code,trade_date,open,high,low,close,pre_close,change,pct_chg,vol,amount'
)

# 按日期升序排列
df_a = df_a.sort_values('trade_date').reset_index(drop=True)
print(f'A股数据获取完成：{len(df_a)} 条记录')
print(f'区间: {df_a["trade_date"].iloc[0]} ~ {df_a["trade_date"].iloc[-1]}')
print(f'最新收盘价: ¥{df_a["close"].iloc[-1]}')

In [ ]:
# 预览 A 股数据
df_a.head(10)

In [ ]:
# A 股基本统计
df_a[['open','high','low','close','vol','amount']].describe()

## 四、获取中芯国际港股日线数据

数据来源：**腾讯股票行情 API**（前复权日线）  
> Tushare 的 `hk_daily` 接口有频率限制（1次/小时），这里使用腾讯行情 API 作为替代。

API 格式：`https://web.ifzq.gtimg.cn/appstock/app/hkfqkline/get?param=hk00981,day,起始,结束,640,qfq`

In [ ]:
def fetch_hk_data():
    """通过腾讯行情API获取港股前复权日线数据"""
    start_fmt = f'{START_DATE[:4]}-{START_DATE[4:6]}-{START_DATE[6:]}'
    end_fmt = f'{END_DATE[:4]}-{END_DATE[4:6]}-{END_DATE[6:]}'
    url = f'https://web.ifzq.gtimg.cn/appstock/app/hkfqkline/get?param=hk00981,day,{start_fmt},{end_fmt},640,qfq'
    
    # 禁用代理，避免网络问题
    session = requests.Session()
    session.trust_env = False
    session.proxies = {'http': None, 'https': None}
    
    resp = session.get(url, timeout=15, headers={
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
        'Referer': 'https://web.ifzq.gtimg.cn/',
    })
    data = resp.json()
    
    stock_data = data.get('data', {}).get('hk00981', {})
    klines = stock_data.get('qfqday') or stock_data.get('day') or []
    
    if not klines:
        print('港股数据获取失败')
        return None
    
    # 解析: [date, open, close, high, low, volume, {}, pct_chg, amount]
    records = []
    for k in klines:
        close = float(k[2])
        open_ = float(k[1])
        records.append({
            'ts_code': '0981.HK',
            'trade_date': k[0].replace('-', ''),
            'open': open_,
            'close': close,
            'high': float(k[3]),
            'low': float(k[4]),
            'vol': float(k[5]) / 100,      # 股 -> 手
            'amount': float(k[8]) * 10000 if len(k) > 8 and k[8] else 0,
            'pct_chg': float(k[7]) if len(k) > 7 and k[7] else 0,
            'change': close - open_,
        })
    
    df = pd.DataFrame(records)
    df['pre_close'] = df['close'].shift(1)
    df.loc[0, 'pre_close'] = df.loc[0, 'close'] - df.loc[0, 'change']
    return df

df_hk = fetch_hk_data()
print(f'港股数据获取完成：{len(df_hk)} 条记录')
print(f'区间: {df_hk["trade_date"].iloc[0]} ~ {df_hk["trade_date"].iloc[-1]}')
print(f'最新收盘价: HK${df_hk["close"].iloc[-1]}')

In [ ]:
# 预览港股数据
df_hk.head(10)

## 五、数据对齐与指标计算

A股和港股交易日历不完全重合，需要取**共同交易日**进行对齐。  
同时计算 **AH 溢价率**：

$$AH溢价率 = \frac{A股价格 - 港股价格 \times 汇率}{港股价格 \times 汇率} \times 100\%$$

In [ ]:
# 取共同交易日
dates_a = set(df_a['trade_date'].tolist())
dates_hk = set(df_hk['trade_date'].tolist())
common_dates = sorted(dates_a & dates_hk)
print(f'A股交易日: {len(dates_a)} 天')
print(f'港股交易日: {len(dates_hk)} 天')
print(f'共同交易日: {len(common_dates)} 天')

# 筛选共同交易日
df_a_common = df_a[df_a['trade_date'].isin(common_dates)].copy().reset_index(drop=True)
df_hk_common = df_hk[df_hk['trade_date'].isin(common_dates)].copy().reset_index(drop=True)

# 计算AH溢价率
df_hk_cny = df_hk_common['close'] * HKD_CNY_RATE
ah_premium = ((df_a_common['close'].values - df_hk_cny.values) / df_hk_cny.values * 100)

# 构建对比DataFrame
df_compare = pd.DataFrame({
    'trade_date': df_a_common['trade_date'],
    'a_open': df_a_common['open'].values,
    'a_high': df_a_common['high'].values,
    'a_low': df_a_common['low'].values,
    'a_close': df_a_common['close'].values,
    'a_vol': df_a_common['vol'].values,
    'a_pct': df_a_common['pct_chg'].values,
    'hk_open': df_hk_common['open'].values,
    'hk_high': df_hk_common['high'].values,
    'hk_low': df_hk_common['low'].values,
    'hk_close': df_hk_common['close'].values,
    'hk_close_cny': df_hk_cny.values,
    'hk_vol': df_hk_common['vol'].values,
    'hk_pct': df_hk_common['pct_chg'].values,
    'ah_premium': ah_premium,
})

# 日期格式转换
df_compare['date'] = pd.to_datetime(df_compare['trade_date'], format='%Y%m%d')

# 归一化价格（首日=100）
df_compare['a_norm'] = df_compare['a_close'] / df_compare['a_close'].iloc[0] * 100
df_compare['hk_norm'] = df_compare['hk_close'] / df_compare['hk_close'].iloc[0] * 100

df_compare.head(10)

## 六、K 线图绘制

使用 **mplfinance** 绘制专业 K 线图，配色遵循中国股市惯例（涨红跌绿）。
### 6.1 中芯国际 A 股 K 线图

In [ ]:
# 准备 mplfinance 所需格式
df_a_plot = df_a_common.copy()
df_a_plot['date'] = pd.to_datetime(df_a_plot['trade_date'], format='%Y%m%d')
df_a_plot = df_a_plot.set_index('date')
df_a_plot = df_a_plot[['open', 'high', 'low', 'close', 'vol']]
df_a_plot.columns = ['Open', 'High', 'Low', 'Close', 'Volume']

# 自定义配色（涨红跌绿）
mc = mpf.make_marketcolors(
    up=UP_COLOR, down=DOWN_COLOR,
    edge=UP_COLOR, edgedown=DOWN_COLOR,
    wick=UP_COLOR, wickdown=DOWN_COLOR,
    volume={'up': UP_COLOR, 'down': DOWN_COLOR},
    inherit=True
)
style_custom = mpf.make_mpf_style(
    base_mpf_style='dark_background',
    marketcolors=mc,
    rc={'font.sans-serif': 'SimHei', 'axes.unicode_minus': False}
)

# 绘制 A 股 K 线 + 成交量
fig, axes = mpf.plot(
    df_a_plot,
    type='candle',
    style=style_custom,
    title='中芯国际 A股 688981.SH K线图',
    ylabel='价格 (CNY)',
    ylabel_lower='成交量 (手)',
    volume=True,
    figsize=(16, 8),
    returnfig=True
)
plt.show()

### 6.2 中芯国际港股 K 线图

In [ ]:
# 港股数据准备
df_hk_plot = df_hk_common.copy()
df_hk_plot['date'] = pd.to_datetime(df_hk_plot['trade_date'], format='%Y%m%d')
df_hk_plot = df_hk_plot.set_index('date')
df_hk_plot = df_hk_plot[['open', 'high', 'low', 'close', 'vol']]
df_hk_plot.columns = ['Open', 'High', 'Low', 'Close', 'Volume']

# 绘制港股 K 线 + 成交量
fig, axes = mpf.plot(
    df_hk_plot,
    type='candle',
    style=style_custom,
    title='中芯国际 港股 00981.HK K线图',
    ylabel='价格 (HKD)',
    ylabel_lower='成交量 (手)',
    volume=True,
    figsize=(16, 8),
    returnfig=True
)
plt.show()

## 七、归一化价格走势对比

将两个市场的收盘价归一化为首日 = 100，直观对比涨跌幅度。

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))
fig.patch.set_facecolor('#0f1117')
ax.set_facecolor('#1a1d28')

ax.plot(df_compare['date'], df_compare['a_norm'],
        color=UP_COLOR, linewidth=2, label='A股 688981.SH', alpha=0.9)
ax.plot(df_compare['date'], df_compare['hk_norm'],
        color=DOWN_COLOR, linewidth=2, label='港股 00981.HK', alpha=0.9)

ax.axhline(y=100, color='#555', linestyle='--', linewidth=0.8, alpha=0.5)

ax.set_title('中芯国际 A股 vs 港股 归一化价格走势 (首日=100)', fontsize=16, color='#e0e0e0', pad=15)
ax.set_ylabel('归一化价格', fontsize=12, color='#c0c4d0')
ax.legend(fontsize=12, facecolor='#252836', edgecolor='#3a3d4a', labelcolor='#e0e0e0')
ax.tick_params(colors='#8b8fa3')
ax.grid(True, color='#2a2d3a', linewidth=0.5, alpha=0.5)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator())
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 八、成交量对比

柱状图配色跟随当日涨跌：涨红跌绿。

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10), sharex=True)
fig.patch.set_facecolor('#0f1117')

# A股成交量
a_colors = [UP_COLOR if df_compare['a_close'].iloc[i] >= df_a_common['pre_close'].iloc[i] else DOWN_COLOR
            for i in range(len(df_compare))]
ax1.set_facecolor('#1a1d28')
ax1.bar(df_compare['date'], df_compare['a_vol'], color=a_colors, width=0.8, alpha=0.8)
ax1.set_title('A股 688981.SH 成交量', fontsize=14, color='#e0e0e0', pad=10)
ax1.set_ylabel('成交量 (手)', fontsize=11, color='#c0c4d0')
ax1.tick_params(colors='#8b8fa3')
ax1.grid(True, color='#2a2d3a', linewidth=0.5, alpha=0.5)

# 港股成交量
hk_colors = [UP_COLOR if df_compare['hk_close'].iloc[i] >= df_hk_common['pre_close'].iloc[i] else DOWN_COLOR
             for i in range(len(df_compare))]
ax2.set_facecolor('#1a1d28')
ax2.bar(df_compare['date'], df_compare['hk_vol'], color=hk_colors, width=0.8, alpha=0.8)
ax2.set_title('港股 00981.HK 成交量', fontsize=14, color='#e0e0e0', pad=10)
ax2.set_ylabel('成交量 (手)', fontsize=11, color='#c0c4d0')
ax2.set_xlabel('日期', fontsize=11, color='#c0c4d0')
ax2.tick_params(colors='#8b8fa3')
ax2.grid(True, color='#2a2d3a', linewidth=0.5, alpha=0.5)
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax2.xaxis.set_major_locator(mdates.MonthLocator())
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 九、AH 溢价率走势

AH 溢价率反映 A 股相对港股的估值差异：
- **正值**：A 股溢价（A 股更贵）
- **负值**：A 股折价（港股更贵）

In [ ]:
fig, ax = plt.subplots(figsize=(16, 5))
fig.patch.set_facecolor('#0f1117')
ax.set_facecolor('#1a1d28')

ax.fill_between(df_compare['date'], df_compare['ah_premium'], 0,
                where=df_compare['ah_premium'] >= 0,
                color='#60a5fa', alpha=0.15, interpolate=True)
ax.plot(df_compare['date'], df_compare['ah_premium'],
        color='#60a5fa', linewidth=2, label='AH溢价率')

# 均值线
prem_avg = df_compare['ah_premium'].mean()
ax.axhline(y=prem_avg, color='#fbbf24', linestyle='--', linewidth=1.2,
           label=f'均值 {prem_avg:.2f}%')
ax.axhline(y=0, color='#555', linewidth=0.8)

ax.set_title('中芯国际 AH 溢价率走势', fontsize=16, color='#e0e0e0', pad=15)
ax.set_ylabel('溢价率 (%)', fontsize=12, color='#c0c4d0')
ax.set_xlabel('日期', fontsize=11, color='#c0c4d0')
ax.legend(fontsize=12, facecolor='#252836', edgecolor='#3a3d4a', labelcolor='#e0e0e0')
ax.tick_params(colors='#8b8fa3')
ax.grid(True, color='#2a2d3a', linewidth=0.5, alpha=0.5)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator())
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 十、A 股 vs 港股日涨跌幅散点对比

观察两个市场日收益率的相关性。

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
fig.patch.set_facecolor('#0f1117')
ax.set_facecolor('#1a1d28')

ax.scatter(df_compare['a_pct'], df_compare['hk_pct'],
           c='#60a5fa', alpha=0.6, s=30, edgecolors='#3a3d4a')

# 回归线
z = np.polyfit(df_compare['a_pct'], df_compare['hk_pct'], 1)
p = np.poly1d(z)
x_range = np.linspace(df_compare['a_pct'].min(), df_compare['a_pct'].max(), 100)
ax.plot(x_range, p(x_range), color=UP_COLOR, linewidth=2, label=f'回归线 (斜率={z[0]:.3f})')

# 相关系数
corr = df_compare['a_pct'].corr(df_compare['hk_pct'])

ax.set_title(f'A股 vs 港股 日涨跌幅 (相关系数={corr:.3f})', fontsize=14, color='#e0e0e0', pad=15)
ax.set_xlabel('A股日涨跌幅 (%)', fontsize=12, color='#c0c4d0')
ax.set_ylabel('港股日涨跌幅 (%)', fontsize=12, color='#c0c4d0')
ax.axhline(y=0, color='#555', linewidth=0.5)
ax.axvline(x=0, color='#555', linewidth=0.5)
ax.legend(fontsize=11, facecolor='#252836', edgecolor='#3a3d4a', labelcolor='#e0e0e0')
ax.tick_params(colors='#8b8fa3')
ax.grid(True, color='#2a2d3a', linewidth=0.5, alpha=0.5)
plt.tight_layout()
plt.show()

## 十一、综合统计摘要

In [ ]:
a_ret = (df_compare['a_close'].iloc[-1] / df_compare['a_close'].iloc[0] - 1) * 100
hk_ret = (df_compare['hk_close'].iloc[-1] / df_compare['hk_close'].iloc[0] - 1) * 100

print('=' * 60)
print('           中芯国际 AH 股对比分析摘要')
print('=' * 60)
print(f'  数据区间:        {df_compare["trade_date"].iloc[0]} ~ {df_compare["trade_date"].iloc[-1]}')
print(f'  共同交易日:      {len(df_compare)} 天')
print(f'  汇率 (HKD/CNY):  {HKD_CNY_RATE}')
print('-' * 60)
print(f'  A股最新收盘:     ¥{df_compare["a_close"].iloc[-1]:.2f}')
print(f'  A股区间涨幅:     {a_ret:+.2f}%')
print(f'  A股区间最高:     ¥{df_compare["a_high"].max():.2f}')
print(f'  A股区间最低:     ¥{df_compare["a_low"].min():.2f}')
print(f'  A股日均成交量:   {df_compare["a_vol"].mean():.0f} 手')
print('-' * 60)
print(f'  港股最新收盘:    HK${df_compare["hk_close"].iloc[-1]:.2f} (约 ¥{df_compare["hk_close_cny"].iloc[-1]:.2f})')
print(f'  港股区间涨幅:    {hk_ret:+.2f}%')
print(f'  港股区间最高:    HK${df_compare["hk_high"].max():.2f}')
print(f'  港股区间最低:    HK${df_compare["hk_low"].min():.2f}')
print(f'  港股日均成交量:  {df_compare["hk_vol"].mean():.0f} 手')
print('-' * 60)
print(f'  AH溢价率(最新):  {df_compare["ah_premium"].iloc[-1]:.2f}%')
print(f'  AH溢价率(均值):  {df_compare["ah_premium"].mean():.2f}%')
print(f'  AH溢价率(最高):  {df_compare["ah_premium"].max():.2f}%')
print(f'  AH溢价率(最低):  {df_compare["ah_premium"].min():.2f}%')
print('-' * 60)
print(f'  日涨跌幅相关系数: {corr:.4f}')
print(f'  港股涨幅领先A股:  {hk_ret - a_ret:+.2f} 个百分点')
print('=' * 60)

## 十二、保存数据到本地

将获取的数据保存为 CSV 和 JSON 格式，方便后续分析。

In [ ]:
# 保存 A 股数据
df_a.to_csv('smic_a_share.csv', index=False, encoding='utf-8-sig')
print(f'A股数据已保存: smic_a_share.csv ({len(df_a)} 条)')

# 保存港股数据
df_hk.to_csv('smic_hk_share.csv', index=False, encoding='utf-8-sig')
print(f'港股数据已保存: smic_hk_share.csv ({len(df_hk)} 条)')

# 保存对比数据
df_compare.to_csv('smic_comparison.csv', index=False, encoding='utf-8-sig')
print(f'对比数据已保存: smic_comparison.csv ({len(df_compare)} 条)')
print('\n所有数据保存完成！')